# 06 — CLV, Prioritization & ROI Simulation

**Inputs:** `telco_features_scored_latest.parquet` (Module 3). Optional, used only for sensitivity checks: `telco_survival_approximation_latest.parquet` (Module 4) and `telco_uplift_experimental_latest.parquet` (Module 5).

This notebook covers **Module 6** (CLV × risk prioritization) and **Module 7** (revenue-at-risk and ROI simulation) of the proposal in one place, as laid out in the repository structure. It is part of the **core system**: unlike Modules 4 and 5, it does not depend on any future-work output.

**The question it answers** (proposal Section 2): *given a limited marketing budget, which customers should we target, and how much revenue can we realistically save?*

**Scope of this notebook**
1. Obtain honest (out-of-fold) churn probabilities
2. State the business assumptions the data cannot supply
3. Estimate customer lifetime value (CLV), and test how much the CLV method matters
4. Size the revenue and value at risk
5. Derive the expected net value of contacting each customer, including a closed-form break-even risk
6. Build the CLV × risk prioritization matrix
7. Simulate campaigns: profit curves, profit-optimal vs. accuracy/F1-optimal thresholds, business KPIs, break-even save rates, sensitivity, and a robust contact list
8. Persist a ranked target list and the assumptions behind it


## What This Module Can and Cannot Claim

**What it is built on.** Real customer data and Module 3's churn model, combined with **explicit business assumptions** that the IBM dataset cannot supply: profit margin, the cost and depth of an offer, how many would-be churners an offer actually saves, the CLV horizon, and the discount rate. Every one of these is a labeled placeholder in Section 3, not a finding.

**What it can claim**
- *Relative* statements: who should be prioritized, how much value-aware ranking adds over risk-only ranking, how profit-optimal thresholds differ from accuracy- or F1-optimal ones
- *Conditional* statements: "under these assumptions, the profit-optimal campaign contacts about X% of customers"
- The **shape** of the economics: how required save rates rise with campaign size, and which assumptions matter most

**What it cannot claim**
- Absolute dollar figures as forecasts. They scale directly with the assumptions (Section 8.5 shows profit moving by roughly a factor of 10 across plausible ranges)
- That an offer will save any particular share of churners. That number can only come from a randomized pilot (Module 5 sizes one)

**What it deliberately does not depend on.** The core path uses Modules 1–3 only. Module 4's survival model appears only as a CLV *sensitivity check*, and Module 5's simulated uplift appears only in an optional cross-tabulation.


In [1]:
import json
import warnings
from datetime import date
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (accuracy_score, average_precision_score, brier_score_loss,
                             f1_score, roc_auc_score)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

FEATURE_STORE_DIR = Path("../data/feature_store")
CONFIGS_DIR = Path("../configs")
REPORTS_DIR = Path("../reports")
CONFIGS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


## 2. Load Data and Obtain Out-of-Fold Churn Probabilities

In [2]:
df = pd.read_parquet(FEATURE_STORE_DIR / "telco_features_scored_latest.parquet")
n = len(df)
y = (df["Churn"] == "Yes").astype(int).to_numpy()
mc = df["MonthlyCharges"].to_numpy()
print(f"{n:,} customers | observed churn rate {y.mean():.4f} | mean monthly charges ${mc.mean():.2f}")
df[["customerID", "persona_name", "tenure", "MonthlyCharges", "churn_probability"]].head(3)


7,043 customers | observed churn rate 0.2654 | mean monthly charges $64.76


,customerID,persona_name,tenure,MonthlyCharges,churn_probability
0,7590-VHVEG,New & Price-Sensitive,1,29.85,0.640019
1,5575-GNVDE,New & Price-Sensitive,34,56.95,0.054867
2,3668-QPYBK,New & Price-Sensitive,2,53.85,0.405564


### 2.1 Why not simply use the Module 3 scores?

Module 3's scored dataset scores *every* customer with a model that was trained on 80% of them. For those customers the scores are **in-sample** and look sharper than they will on customers the model has never seen. Business totals built on over-sharp scores would overstate how well targeting works.

Here every customer is instead scored by a model that **never saw them**: 5-fold cross-fitting with the same preprocessing, XGBoost settings, and isotonic calibration as Module 3. (`n_jobs=1` is used for determinism.)


In [3]:
numeric_features = ["tenure", "MonthlyCharges", "services_count", "contract_risk_flag"]
categorical_features = [
    "gender", "SeniorCitizen", "Partner", "Dependents",
    "PhoneService", "MultipleLines", "InternetService",
    "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport",
    "StreamingTV", "StreamingMovies", "Contract", "PaperlessBilling", "PaymentMethod",
]
X_all = df[numeric_features + categorical_features]

p_oof = np.zeros(n)
for tr, te in StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED).split(X_all, y):
    pre = ColumnTransformer([
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), categorical_features),
    ])
    X_tr, X_te = pre.fit_transform(X_all.iloc[tr]), pre.transform(X_all.iloc[te])
    base = XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        scale_pos_weight=(y[tr] == 0).sum() / (y[tr] == 1).sum(),
        eval_metric="logloss", random_state=RANDOM_SEED, n_jobs=1,
    )
    calibrated = CalibratedClassifierCV(estimator=base, method="isotonic", cv=5).fit(X_tr, y[tr])
    p_oof[te] = calibrated.predict_proba(X_te)[:, 1]

p_m3 = df["churn_probability"].to_numpy()
comparison = pd.DataFrame({
    "ROC-AUC": [roc_auc_score(y, p_m3), roc_auc_score(y, p_oof)],
    "PR-AUC": [average_precision_score(y, p_m3), average_precision_score(y, p_oof)],
    "Brier score": [brier_score_loss(y, p_m3), brier_score_loss(y, p_oof)],
    "mean probability": [p_m3.mean(), p_oof.mean()],
    "share with p >= 0.5": [(p_m3 >= 0.5).mean(), (p_oof >= 0.5).mean()],
}, index=["Module 3 scores (partly in-sample)", "Out-of-fold scores (used below)"]).round(4)
print(f"Observed churn rate for reference: {y.mean():.4f}")
comparison


Observed churn rate for reference: 0.2654


,ROC-AUC,PR-AUC,Brier score,mean probability,share with p >= 0.5
Module 3 scores (partly in-sample),0.8933,0.7415,0.1168,0.2665,0.1966
Out-of-fold scores (used below),0.8412,0.6525,0.1367,0.2649,0.1778


The out-of-fold ROC-AUC matches Module 3's held-out result (about 0.84), while the all-customer Module 3 score looks markedly better because most of those customers were in its training set. The out-of-fold mean probability also lines up with the observed churn rate, confirming the calibration carries over. **All business calculations below use the out-of-fold probability**, referred to as `p`.


In [4]:
p = p_oof
df["churn_probability_oof"] = p
